# Tugas Besar Fundamen Sains Data
## Analisis Data Nutrisi Makanan: Unsupervised & Supervised Learning

Notebook ini mencakup:
1. Data Understanding & Cleaning (EDA)
2. Preprocessing (missing value, encoding, scaling)
3. Dimensionality Reduction (PCA)
4. Unsupervised Learning: K-Means, Hierarchical Clustering, DBSCAN
5. Supervised Learning: Random Forest
6. Evaluasi & Kesimpulan

> Dataset: [Nutritional values for common foods and products](https://www.kaggle.com/datasets/trolukovich/nutritional-values-for-common-foods-and-products) (Kaggle, ~8.8k baris, 76 kolom nutrisi per 100g + kolom `name` & `serving_size`).
>
> **Catatan penting (sudah diverifikasi dari file asli):**
> - Dataset **tidak punya kolom kategori bawaan** — kategori diturunkan dari kata pertama sebelum koma pada kolom `name` (mis. `"Nuts, pecans"` → `"Nuts"`), lihat Bagian 1.1.
> - Hampir semua nilai nutrisi tersimpan sebagai **teks bersatuan** (mis. `"0.1g"`, `"9.00 mg"`, `"22.00 mcg"`, `"56.00 IU"`), bukan angka murni — harus dibersihkan dulu (lihat Bagian 1.2) sebelum dipakai untuk EDA/PCA/clustering/Random Forest.
> - Kolom nutrisi lengkap (76 kolom): `serving_size, calories, total_fat, saturated_fat, cholesterol, sodium, choline, folate, folic_acid, niacin, pantothenic_acid, riboflavin, thiamin, vitamin_a, vitamin_a_rae, carotene_alpha, carotene_beta, cryptoxanthin_beta, lutein_zeaxanthin, lucopene, vitamin_b12, vitamin_b6, vitamin_c, vitamin_d, vitamin_e, tocopherol_alpha, vitamin_k, calcium, copper, irom, magnesium, manganese, phosphorous, potassium, selenium, zink, protein, alanine, arginine, aspartic_acid, cystine, glutamic_acid, glycine, histidine, hydroxyproline, isoleucine, leucine, lysine, methionine, phenylalanine, proline, serine, threonine, tryptophan, tyrosine, valine, carbohydrate, fiber, sugars, fructose, galactose, glucose, lactose, maltose, sucrose, fat, saturated_fatty_acids, monounsaturated_fatty_acids, polyunsaturated_fatty_acids, fatty_acids_total_trans, alcohol, ash, caffeine, theobromine, water`. (Catatan: beberapa nama memang bertypo di sumber aslinya — `irom` = iron, `zink` = zinc, `lucopene` = lycopene.)
> - File CSV punya kolom index tanpa nama di paling depan — sudah ditangani dengan `index_col=0` saat load data.

## 0. Konfigurasi & Import Library

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay
)
from scipy.cluster.hierarchy import dendrogram, linkage

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)

RANDOM_STATE = 42

In [ ]:
# ==== KONFIGURASI ====
DATA_PATH = "data/nutrition.csv"   # path file dataset (nutrition.csv dari Kaggle trolukovich)
TARGET_COL = "food_category"        # kolom label hasil turunan dari kolom 'name' (lihat Bagian 1)
ID_COLS = ["name", "serving_size"]  # kolom non-numerik/identitas yang di-drop sebelum modeling
                                     # 'serving_size' bertipe string (mis. "100 g") makanya di-drop

## 1. Data Understanding & Cleaning (EDA)

In [ ]:
df = pd.read_csv(DATA_PATH, index_col=0)  # kolom pertama file ini adalah index tanpa nama
print(df.shape)
print(df.columns.tolist())  # cek nama kolom asli file CSV-mu, cocokkan dengan yang dipakai di notebook ini
df.head()

### 1.1 Membuat Kolom Kategori (`food_category`)

Dataset ini tidak punya kolom kategori bawaan. Kolom `name` berformat `"Jenis, detail"` (contoh: `"Cheese, cheddar"`), jadi kategori diambil dari kata pertama sebelum koma. Kategori yang jumlah datanya terlalu sedikit digabung jadi `"Other"` supaya Random Forest tidak kesulitan belajar dari kelas yang nyaris kosong.

In [ ]:
# Turunkan kategori dari kata pertama sebelum koma pada kolom 'name'
df["food_category"] = (
    df["name"].astype(str).str.split(",").str[0].str.strip().str.title()
)

MIN_CATEGORY_COUNT = 30  # ambang minimal jumlah baris per kategori
counts = df["food_category"].value_counts()
rare_categories = counts[counts < MIN_CATEGORY_COUNT].index
df.loc[df["food_category"].isin(rare_categories), "food_category"] = "Other"

print(f"Jumlah kategori setelah digabung: {df['food_category'].nunique()}")
df["food_category"].value_counts().head(20)

### 1.2 Membersihkan Kolom Nutrisi (Konversi ke Numerik)

Nilai pada hampir semua kolom nutrisi tersimpan sebagai teks dengan satuan menempel (contoh: `"0.1g"`, `"9.00 mg"`, `"22.00 mcg"`, `"56.00 IU"`), bukan angka murni — jadi harus dibersihkan dulu sebelum bisa dipakai untuk EDA, PCA, clustering, maupun Random Forest.

In [ ]:
def parse_numeric(series):
    """Ekstrak angka di awal string dan buang satuan (g, mg, mcg, IU, dll)."""
    extracted = series.astype(str).str.extract(r"(-?\d+\.?\d*)")[0]
    return pd.to_numeric(extracted, errors="coerce")

# Semua kolom kecuali identitas/teks dianggap kolom nutrisi yang perlu dibersihkan
non_nutrient_cols = {"name", "serving_size", "food_category"}
nutrient_cols = [c for c in df.columns if c not in non_nutrient_cols]

for col in nutrient_cols:
    df[col] = parse_numeric(df[col])

print("Jumlah kolom nutrisi yang dikonversi ke numerik:", len(nutrient_cols))
df[nutrient_cols].dtypes.value_counts()

In [ ]:
df.info()

In [ ]:
# Cek missing value & duplikat
print("Missing value per kolom:")
print(df.isna().sum().sort_values(ascending=False).head(15))
print("\nJumlah baris duplikat:", df.duplicated().sum())

In [ ]:
# Statistik deskriptif fitur numerik
df.describe().T

In [ ]:
# Distribusi target (kategori makanan)
if TARGET_COL in df.columns:
    df[TARGET_COL].value_counts().plot(kind="bar", figsize=(10, 4))
    plt.title("Distribusi Kategori Makanan")
    plt.ylabel("Jumlah")
    plt.show()

In [ ]:
# Korelasi antar fitur numerik
numeric_df = df.select_dtypes(include=[np.number])
plt.figure(figsize=(12, 9))
sns.heatmap(numeric_df.corr(), cmap="coolwarm", center=0)
plt.title("Heatmap Korelasi Fitur Nutrisi")
plt.show()

## 2. Preprocessing

In [ ]:
# Bersihkan data: drop kolom ID, tangani missing value
clean_df = df.drop(columns=[c for c in ID_COLS if c in df.columns]).copy()

# Isi missing value numerik dengan median (aman terhadap outlier)
num_cols = clean_df.select_dtypes(include=[np.number]).columns.tolist()
for col in num_cols:
    clean_df[col] = clean_df[col].fillna(clean_df[col].median())

# Drop baris yang label targetnya kosong (kalau ada)
if TARGET_COL in clean_df.columns:
    clean_df = clean_df.dropna(subset=[TARGET_COL])

clean_df.shape

In [ ]:
# Pisahkan fitur (X) dan target (y)
feature_cols = [c for c in num_cols if c != TARGET_COL]
X = clean_df[feature_cols]

y = None
le = None
if TARGET_COL in clean_df.columns:
    le = LabelEncoder()
    y = le.fit_transform(clean_df[TARGET_COL])

# Standardisasi fitur (wajib sebelum PCA & clustering berbasis jarak)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=feature_cols)
X_scaled.head()

## 3. Dimensionality Reduction (PCA)

In [ ]:
# Cari jumlah komponen yang menjelaskan ~90-95% variansi
pca_full = PCA(random_state=RANDOM_STATE).fit(X_scaled)
cum_var = np.cumsum(pca_full.explained_variance_ratio_)

plt.plot(range(1, len(cum_var) + 1), cum_var, marker="o")
plt.axhline(0.9, color="red", linestyle="--", label="90% variansi")
plt.xlabel("Jumlah Komponen")
plt.ylabel("Kumulatif Explained Variance")
plt.title("Scree Plot PCA")
plt.legend()
plt.show()

n_components_90 = np.argmax(cum_var >= 0.90) + 1
print(f"Jumlah komponen untuk >=90% variansi: {n_components_90}")

In [ ]:
# PCA untuk visualisasi (2D)
pca_2d = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca_2d = pca_2d.fit_transform(X_scaled)

# PCA untuk dipakai di clustering/model (menjaga 90% variansi)
pca_reduced = PCA(n_components=n_components_90, random_state=RANDOM_STATE)
X_pca_reduced = pca_reduced.fit_transform(X_scaled)

print("Shape data setelah PCA (2D):", X_pca_2d.shape)
print("Shape data setelah PCA (reduced):", X_pca_reduced.shape)

## 4. Unsupervised Learning: Clustering

### 4.1 K-Means

In [ ]:
# Elbow method + silhouette score untuk menentukan k optimal
inertias, sil_scores = [], []
k_range = range(2, 11)

for k in k_range:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels = km.fit_predict(X_pca_reduced)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_pca_reduced, labels))

fig, ax = plt.subplots(1, 2, figsize=(14, 4))
ax[0].plot(k_range, inertias, marker="o")
ax[0].set_title("Elbow Method")
ax[0].set_xlabel("k")
ax[0].set_ylabel("Inertia")

ax[1].plot(k_range, sil_scores, marker="o", color="green")
ax[1].set_title("Silhouette Score")
ax[1].set_xlabel("k")
ax[1].set_ylabel("Silhouette Score")
plt.show()

In [ ]:
# Set k terbaik berdasarkan grafik di atas, lalu fit final model
K_OPTIMAL = 4  # TODO: ganti sesuai hasil elbow/silhouette

kmeans_final = KMeans(n_clusters=K_OPTIMAL, random_state=RANDOM_STATE, n_init=10)
kmeans_labels = kmeans_final.fit_predict(X_pca_reduced)

plt.scatter(X_pca_2d[:, 0], X_pca_2d[:, 1], c=kmeans_labels, cmap="tab10", s=15)
plt.title(f"K-Means Clustering (k={K_OPTIMAL}) - proyeksi PCA 2D")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.show()

### 4.2 Hierarchical Clustering

In [ ]:
# Dendrogram (pakai sampel jika data besar, agar tidak berat)
sample_size = min(200, X_pca_reduced.shape[0])
sample_idx = np.random.RandomState(RANDOM_STATE).choice(
    X_pca_reduced.shape[0], sample_size, replace=False
)
X_sample = X_pca_reduced[sample_idx]

linked = linkage(X_sample, method="ward")
plt.figure(figsize=(14, 5))
dendrogram(linked, truncate_mode="lastp", p=30)
plt.title("Dendrogram (Ward Linkage, sampel data)")
plt.xlabel("Sampel")
plt.ylabel("Jarak")
plt.show()

In [ ]:
agglo = AgglomerativeClustering(n_clusters=K_OPTIMAL, linkage="ward")
agglo_labels = agglo.fit_predict(X_pca_reduced)

plt.scatter(X_pca_2d[:, 0], X_pca_2d[:, 1], c=agglo_labels, cmap="tab10", s=15)
plt.title(f"Hierarchical Clustering (k={K_OPTIMAL}) - proyeksi PCA 2D")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.show()

### 4.3 DBSCAN

In [ ]:
# Cari eps yang masuk akal pakai k-distance plot (k = min_samples)
from sklearn.neighbors import NearestNeighbors

MIN_SAMPLES = 5
neighbors = NearestNeighbors(n_neighbors=MIN_SAMPLES).fit(X_pca_reduced)
distances, _ = neighbors.kneighbors(X_pca_reduced)
k_distances = np.sort(distances[:, -1])

plt.plot(k_distances)
plt.title("K-Distance Plot (untuk menentukan eps)")
plt.xlabel("Titik data (terurut)")
plt.ylabel(f"Jarak ke tetangga ke-{MIN_SAMPLES}")
plt.show()

In [ ]:
EPS = 1.5  # TODO: ganti sesuai titik "siku" pada k-distance plot

dbscan = DBSCAN(eps=EPS, min_samples=MIN_SAMPLES)
dbscan_labels = dbscan.fit_predict(X_pca_reduced)

n_clusters = len(set(dbscan_labels)) - (1 if -1 in dbscan_labels else 0)
n_noise = list(dbscan_labels).count(-1)
print(f"Jumlah cluster: {n_clusters}, jumlah noise: {n_noise}")

plt.scatter(X_pca_2d[:, 0], X_pca_2d[:, 1], c=dbscan_labels, cmap="tab10", s=15)
plt.title("DBSCAN Clustering - proyeksi PCA 2D (label -1 = noise)")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.show()

### 4.4 Perbandingan Evaluasi Clustering

In [ ]:
def safe_eval(X, labels, name):
    mask = labels != -1  # exclude noise DBSCAN saat evaluasi
    if len(set(labels[mask])) < 2:
        return {"Metode": name, "Silhouette": None, "Davies-Bouldin": None}
    return {
        "Metode": name,
        "Silhouette": silhouette_score(X[mask], labels[mask]),
        "Davies-Bouldin": davies_bouldin_score(X[mask], labels[mask]),
    }

results = [
    safe_eval(X_pca_reduced, kmeans_labels, "K-Means"),
    safe_eval(X_pca_reduced, agglo_labels, "Hierarchical"),
    safe_eval(X_pca_reduced, dbscan_labels, "DBSCAN"),
]
pd.DataFrame(results)

## 5. Supervised Learning: Random Forest

In [ ]:
# Split data train/test (pakai fitur asli yang sudah discaling, bukan hasil PCA,
# supaya feature importance tetap bisa diinterpretasi per nutrisi)
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

rf = RandomForestClassifier(
    n_estimators=300,
    random_state=RANDOM_STATE,
    class_weight="balanced",
    n_jobs=-1,
)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
print("Akurasi:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred, target_names=le.classes_))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=le.classes_)
fig, ax = plt.subplots(figsize=(10, 10))
disp.plot(ax=ax, xticks_rotation=90, colorbar=False)
plt.title("Confusion Matrix - Random Forest")
plt.show()

In [ ]:
# Feature importance: nutrisi mana yang paling berpengaruh terhadap kategori makanan
importances = pd.Series(rf.feature_importances_, index=feature_cols)
importances.sort_values(ascending=False).head(15).plot(kind="barh")
plt.title("Top 15 Feature Importance - Random Forest")
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# (Opsional) Tuning hyperparameter sederhana
param_grid = {
    "n_estimators": [200, 300, 500],
    "max_depth": [None, 10, 20],
    "min_samples_split": [2, 5],
}
grid = GridSearchCV(
    RandomForestClassifier(random_state=RANDOM_STATE, class_weight="balanced"),
    param_grid, cv=3, n_jobs=-1, scoring="f1_macro"
)
# grid.fit(X_train, y_train)  # aktifkan jika ingin tuning (bisa lama tergantung ukuran data)
# print(grid.best_params_)

## 6. Kesimpulan

Isi bagian ini dengan interpretasi hasil, misalnya:
- Berapa komponen PCA yang cukup untuk merepresentasikan data, dan fitur nutrisi apa yang paling dominan di tiap komponen (lihat `pca_reduced.components_`).
- Metode clustering mana (K-Means / Hierarchical / DBSCAN) yang memberi hasil paling koheren berdasarkan silhouette & davies-bouldin score, dan karakteristik tiap cluster (misal: cluster "tinggi protein rendah lemak").
- Performa Random Forest dalam mengklasifikasikan kategori makanan, fitur nutrisi paling berpengaruh, serta kategori mana yang paling sering tertukar (lihat confusion matrix).
- Insight gabungan: apakah cluster yang ditemukan secara unsupervised sejalan dengan kategori/label asli?